In [1]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, accuracy_score, confusion_matrix

In [13]:
df=pd.read_csv(r"C:\Users\USER\Downloads\insurance.csv")
YES_VALUE='yes'
print('shape of DF',df.shape)
print(f'Missing Values',df.isna().sum())
print(df.head())


shape of DF (1338, 7)
Missing Values age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [14]:
df = df.dropna(subset=["age", "bmi", "children", "smoker", "charges"])
df["smoker"] = (df["smoker"] == YES_VALUE).astype(int)
print(df["smoker"].value_counts())

smoker
0    1064
1     274
Name: count, dtype: int64


In [15]:
reg_features = ["age", "bmi", "children", "smoker"]
clf_features = ["age", "bmi", "children", "charges"]

In [17]:
X_reg = df[reg_features]
y_reg = df["charges"]


In [18]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
reg_model = LinearRegression()
reg_model.fit(Xr_train, yr_train)
r2 = r2_score(yr_test, reg_model.predict(Xr_test))
print("Regression R2:", round(r2, 3))

Regression R2: 0.781


In [19]:
X_clf = df[clf_features]
y_clf = df["smoker"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
clf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_model.fit(Xc_train, yc_train)
yc_pred = clf_model.predict(Xc_test)
print("Classification accuracy:", round(accuracy_score(yc_test, yc_pred), 3))
print("Baseline (always non-smoker):", round(1 - y_clf.mean(), 3))
print(confusion_matrix(yc_test, yc_pred))

Classification accuracy: 0.966
Baseline (always non-smoker): 0.795
[[208   5]
 [  4  51]]


In [20]:
os.makedirs("models", exist_ok=True)

In [22]:
joblib.dump(reg_model, "models/regression_model.joblib")
joblib.dump(clf_model, "models/classification_model.joblib")
reg_loaded = joblib.load("models/regression_model.joblib")
clf_loaded = joblib.load("models/classification_model.joblib")

In [24]:
sample_reg = pd.DataFrame([[35, 27.5, 2, 0]], columns=reg_features)
sample_clf = pd.DataFrame([[35, 27.5, 2, 30000]], columns=clf_features)
print("Predicted charges:", round(reg_loaded.predict(sample_reg)[0], 2))
print("Predicted smoker (1=yes, 0=no):", clf_loaded.predict(sample_clf)[0])

Predicted charges: 6741.69
Predicted smoker (1=yes, 0=no): 1
